# gpu_server on Kaggle / Colab: CoTracker3 (+ optional RoMa v2) for a CPU-only client

This notebook runs the repo's generic **gpu_server** on the notebook GPU. It publishes the
server with a **cloudflared quick tunnel** (no account, no inbound port), so a CPU-only machine
can run the robot pipeline with `--tracker-provider remote`.

**One-time setup (Kaggle)**
1. *Settings → Accelerator*: **GPU T4 x2** (or T4 x1). P100 is not supported by recent torch wheels.
2. *Settings → Internet*: **On** (needed to clone, for torch.hub weights and for the tunnel).
3. *Add-ons → Secrets*:
   * `GPU_SERVER_TOKEN`: any long random string (recommended). Use the same value on the client.
     If it is missing, a token is generated and printed below.
   * `GITHUB_TOKEN`: only if the repo is private.
4. Run all cells. The **tunnel cell** prints the `wss://….trycloudflare.com/v1/ws` URL and the
   client env lines. The last cell keeps the server and tunnel alive for `HOURS`.

**Headless (browser closed):** use *Save Version → Save & Run All (Commit)*. Kaggle then runs
the notebook for up to 12 h. Read the URL from the version's live log, or from
`/kaggle/working/gpu_server_url.txt`.

**Colab** works the same: Secrets live in the key icon (`google.colab.userdata`), the work dir
is `/content`, and the runtime type should be T4 GPU.

**Client side (CPU machine, PowerShell)**
```powershell
$env:GPU_SERVER_URL   = "wss://<random>.trycloudflare.com/v1/ws"
$env:GPU_SERVER_TOKEN = "<same token>"
$env:GPU_SERVER_CODEC = "jpeg"   # ~14 KB/frame instead of 192 KB, same tracking accuracy
python main.py --tracking --tracker-provider remote --tracking-server no-launch ...
```

In [ ]:
# ---- configuration -------------------------------------------------------------------------
REPO     = "https://github.com/dekelcohen/language-models-trajectory-generators.git"
BRANCH   = "tracking"
MODULES  = ["cotracker", "echo"]       # add "romav2" with WITH_ROMA = True
PRELOAD  = ["cotracker"]               # loaded before the URL is published (~1 min first time)
WITH_ROMA = False                      # RoMa v2 dense matcher for re-detection (~1.5 GB weights)
ROMA_SETTING = "turbo"                 # turbo | fast | base | precise
PORT     = 8765                        # loopback only; the tunnel is the public side
HOURS    = 11.5                        # Kaggle sessions end at 12 h

In [ ]:
# ---- secrets ---------------------------------------------------------------------------------
import os, secrets

def get_secret(name):
    try:                                   # Kaggle
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:                                   # Colab
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

TOKEN = get_secret("GPU_SERVER_TOKEN")
TOKEN_FROM_SECRET = bool(TOKEN)
if not TOKEN:
    TOKEN = secrets.token_urlsafe(18)
    print("No GPU_SERVER_TOKEN secret: generated one (printed with the client lines below).")
GITHUB_TOKEN = get_secret("GITHUB_TOKEN")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())
print("work dir:", WORK, "| token from secret:", TOKEN_FROM_SECRET, "| github token:", bool(GITHUB_TOKEN))

In [ ]:
# ---- clone + install -------------------------------------------------------------------------
import subprocess, sys
REPO_DIR = os.path.join(WORK, "lmtg")
url = REPO.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn[standard]",
                "websockets", "einops", "imageio"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.environ.setdefault("TORCH_HOME", os.path.join(WORK, "torch_cache"))
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - check Accelerator")

In [ ]:
# ---- optional: RoMa v2 -----------------------------------------------------------------------
# Not `pip install -e`: RoMaV2 pins a torchvision that would replace Kaggle's CUDA build.
SETS = []
if WITH_ROMA:
    ROMA_DIR = os.path.join(WORK, "RoMaV2")
    if not os.path.isdir(ROMA_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Parskatt/RoMaV2", ROMA_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "rich"], check=True)
    os.environ["ROMAV2_ROOT"] = ROMA_DIR
    SETS += [f"romav2.root={ROMA_DIR}", f"romav2.setting={ROMA_SETTING}"]
    if "romav2" not in MODULES:
        MODULES.append("romav2")
print("modules:", MODULES, "| preload:", PRELOAD, "| sets:", SETS)

In [ ]:
# ---- start the server on loopback (token enforced on every request) ---------------------------
from gpu_server import expose
LOG_DIR = os.path.join(WORK, "gpu_server_logs")
os.makedirs(LOG_DIR, exist_ok=True)
server = expose.start_server(MODULES, PRELOAD, PORT, TOKEN, SETS,
                             log_path=os.path.join(LOG_DIR, "gpu_server.log"))

In [ ]:
# ---- publish: cloudflared quick tunnel + self-check through the public URL ---------------------
tunnel = expose.Tunnel(PORT, log_path=os.path.join(LOG_DIR, "cloudflared.log")).start()
try:
    reply, rtt = expose.check_public(tunnel.ws_url, TOKEN)
    print(f"public URL answers: median ping {rtt:.0f} ms, loaded {reply.get('loaded')}")
except RuntimeError as exc:
    print("WARNING self-check failed (DNS of a new tunnel can lag; try from the client):", exc)

def publish(ws_url):
    with open(os.path.join(WORK, "gpu_server_url.txt"), "w") as f:
        f.write(ws_url + "\n")
    print(expose.client_instructions(ws_url, TOKEN, show_token=not TOKEN_FROM_SECRET))

publish(tunnel.ws_url)

In [ ]:
# ---- keep alive (blocks). Interrupt the kernel to stop; the next cell cleans up ------------------
# Restarts a dead tunnel (the URL then changes; it is re-printed and re-written).
try:
    expose.keep_alive(server, tunnel, TOKEN, HOURS, on_new_url=publish)
except KeyboardInterrupt:
    print("interrupted")

In [ ]:
# ---- stop -------------------------------------------------------------------------------------
tunnel.stop()
server.terminate()
print("stopped")

### Troubleshooting
* **Client `getaddrinfo failed`**: a fresh `trycloudflare.com` name can take ~1 min to resolve.
  Some resolvers cache the early miss. Wait, or run `ipconfig /flushdns` on Windows.
* **`401` / token errors**: the client's `GPU_SERVER_TOKEN` must equal the notebook's.
* **Slow frames**: use `GPU_SERVER_CODEC=jpeg`. The per-frame cost is internet RTT
  (~60–150 ms) plus the model. That is fine for lock-step runs, but marginal for real time.
* **Logs**: `gpu_server_logs/gpu_server.log` and `cloudflared.log` in the work dir.
* Quick tunnels are for testing (no SLA). For a stable host name, use a named Cloudflare
  tunnel, or ngrok with an auth token (WebSockets work on both).